# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [ ]:
import pandas as pd
import plotly.express as px
import math
import plotly.graph_objects as go


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   str    
 1   continent  1704 non-null   str    
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   str    
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), str(3)
memory usage: 135.3 KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: <ArrowStringArray>
['Asia', 'Europe', 'Africa', 'Americas', 'Oceania']
Length: 5, dtype: str
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [9]:
# Task 1
# YOUR CODE HERE

asia = df.loc[(df['continent'] == 'Asia') & (df['year'].isin([2002, 2007]))].copy()
asia['year'] = asia['year'].astype(str)

wide = asia.pivot(index='country', columns='year', values=['gdpPercap', 'lifeExp'])
avg_gain = (wide['lifeExp']['2007'] - wide['lifeExp']['2002']).mean()
print(f"Average life-expectancy gain in Asia, 2002-2007: {avg_gain:.1f} years")

colour_map = {'2002': 'grey', '2007': '#1f77b4'}

fig = px.scatter(
    asia, x='gdpPercap', y='lifeExp', color='year',
    color_discrete_map=colour_map,
    category_orders={'year': ['2002', '2007']},
    hover_name='country', log_x=True,
    labels={'gdpPercap': 'GDP per Capita (USD, log scale)',
            'lifeExp': 'Life Expectancy (years)', 'year': ''},
    template='plotly_white',
)
fig.update_traces(marker=dict(size=10, opacity=0.85))

for country in wide.index:
    fig.add_annotation(
        x=math.log10(wide.loc[country, ('gdpPercap', '2007')]),
        y=wide.loc[country, ('lifeExp', '2007')],
        ax=math.log10(wide.loc[country, ('gdpPercap', '2002')]),
        ay=wide.loc[country, ('lifeExp', '2002')],
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1,
        arrowcolor='rgba(120,120,120,0.6)', text='')

# label notable points directly
for country in ['Japan', 'China', 'Myanmar']:
    r = asia[(asia['country'] == country) & (asia['year'] == '2007')].iloc[0]
    fig.add_annotation(x=math.log10(r['gdpPercap']), y=r['lifeExp'], text=country,
                       showarrow=True, ax=25, ay=-25, arrowhead=0,
                       font=dict(size=12, color='#1f77b4'))

fig.update_layout(
    title='Between 2002 and 2007 every Asian country gained life expectancy — '
          f'{avg_gain:.1f} years on average',
    font=dict(family='Arial', size=13),
    legend=dict(orientation='h', x=0, y=1.06, title=''),
    height=550,
)
fig.show()

Average life-expectancy gain in Asia, 2002-2007: 1.5 years


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [ ]:
# Task 2
# YOUR CODE HERE

d07 = df.loc[df['year'] == 2007].copy()

highlight = 'Asia'
colour_map = {c: ('crimson' if c == highlight else 'lightgrey') for c in d07['continent'].unique()}

fig = px.scatter(
    d07, x='gdpPercap', y='lifeExp', size='pop', size_max=60,
    color='continent', color_discrete_map=colour_map,
    hover_name='country', hover_data={'continent': False},
    log_x=True, opacity=0.75, template='plotly_white',
    labels={'gdpPercap': 'GDP per Capita (USD, log scale)',
            'lifeExp': 'Life Expectancy (years)', 'pop': 'Population', 'continent': ''},
)
fig.update_traces(marker=dict(line=dict(width=0.5, color='white')))

fig.add_annotation(
    x=math.log10(3000), y=48,
    text='<b>Asia</b> spans the whole range:<br>from poor, short-lived Afghanistan<br>'
         'to rich, long-lived Japan —<br>and holds China &amp; India, the two<br>largest populations',
    showarrow=False, align='left',
    font=dict(color='crimson', size=12, family='Arial'),
    bgcolor='white', bordercolor='crimson', borderwidth=1, borderpad=4)

for country, ax, ay in [('Afghanistan', 40, 35), ('China', 50, -35), ('India', -45, -40) , ('Japan', -10, -45)]:
    r = d07[d07['country'] == country].iloc[0]
    fig.add_annotation(x=math.log10(r['gdpPercap']), y=r['lifeExp'], text=country,
                       showarrow=True, ax=ax, ay=ay, arrowhead=1, arrowcolor='crimson',
                       font=dict(color='crimson', size=12, family='Arial'))

fig.update_layout(
    title='Asia is the whole story in one continent: from Afghanistan to Japan, '
          'and home to the two biggest populations',
    font=dict(family='Arial', size=13),
    showlegend=False, height=600,
)
fig.show()